# Diabetic Retinopathy Baseline - Frozen ResNet-50, Primary Five-Band Task

This notebook runs a **provisional baseline only**. It does **not** modify the existing
dataset, split CSV, split report, or audit artifacts, and it does **not** establish
clinical validity.

Label status: the five-band primary mapping is **provisional**. Two merges
(`13. Very Severe NPDR` -> `Severe NPDR` and `12. Advanced PDR` -> `PDR`) are recorded in
`audit_output/pretraining_audit/LABEL_POLICY_DRAFT.md` and are not specialist-validated.

Execution order:
1. Environment setup + dataset-package integration check
2. Optional dependency install (only if the runtime lacks torch/torchvision)
3. Path resolution (Google Drive supported)
4. Smoke test - **must pass**
5. Demo the guard contract without training (safe)
6. Training cell - **guarded**, disabled by default
7. Final test-partition evaluation - **guarded**, separate and explicitly invoked

Prerequisite for a real run: Runtime -> Change runtime type -> Hardware accelerator -> GPU.
Nothing here claims that GPU execution or training has been verified.


In [ ]:
# 0. Environment setup + dataset-package integration check
import json
import os
import sys
from pathlib import Path

# Make the uploaded project importable.
for hint in (Path("/content/major_project"), Path.cwd()):
    if (hint / "dataset" / "__init__.py").exists():
        if str(hint) not in sys.path:
            sys.path.insert(0, str(hint))
        if str(hint / "colab") not in sys.path:
            sys.path.insert(0, str(hint / "colab"))
        PROJECT_ROOT = hint
        break
else:
    raise RuntimeError(
        "Could not find the uploaded project. Upload it to /content/major_project or run "
        "this notebook from a folder that already contains `dataset/`."
    )

print("project root:", PROJECT_ROOT)
print("python:", sys.version.split()[0])

try:
    import torch
    print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch missing:", exc)

try:
    import torchvision
    print("torchvision:", torchvision.__version__)
except Exception as exc:
    print("torchvision missing:", exc)

# --- Visible integration check against the EXISTING dataset package ---------
# We import the real modules rather than reimplementing their logic.
from dataset.labels import (
    PRIMARY_MAPPING,
    primary_task_label,
    is_primary_eligible_source,
    is_non_dr_source,
)
from dataset.decoding import load_rgb as dataset_load_rgb

import colab_helpers as H

assert H.BAND_ORDER == [
    "No apparent DR",
    "Mild NPDR",
    "Moderate NPDR",
    "Severe NPDR",
    "PDR",
], "colab_helpers.BAND_ORDER drifted from the provisional five-band order"
assert set(PRIMARY_MAPPING.values()) == set(H.BAND_ORDER), (
    "dataset.labels.PRIMARY_MAPPING and colab_helpers.BAND_ORDER disagree"
)
print("dataset.labels import OK; provisional merges in effect:")
print("  13. Very Severe NPDR ->", primary_task_label("13. Very Severe NPDR"))
print("  12. Advanced PDR     ->", primary_task_label("12. Advanced PDR"))
print("non-DR sources are excluded from the primary task:", is_non_dr_source("7.Cataract"))
print("primary-eligible source check:", is_primary_eligible_source("4.Moderate DR"))
print("dataset.decoding.load_rgb import OK:", callable(dataset_load_rgb))
print("colab_helpers.load_rgb delegates to it:", H.load_rgb.__module__)

# Dataset-package integration check complete.
DATASET_PACKAGE_OK = True


In [ ]:
# 1. Optional dependency install - only if the runtime genuinely lacks the stack.
# This never runs automatically; you must decide to run the cell, and it only
# installs from the pinned requirements file that ships with the project.
import subprocess
from pathlib import Path

req_path = PROJECT_ROOT / "colab" / "requirements-colab.txt"

need_install = False
try:
    import torch, torchvision  # noqa: F401
    print("torch/torchvision already present; skipping dependency install.")
except Exception:
    need_install = True

if need_install and req_path.exists():
    print("installing from", req_path)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(req_path)],
        check=True,
    )
elif need_install:
    print("requirements file not found at", req_path, "- install torch/torchvision manually.")


In [ ]:
# 2. Resolve paths (Google Drive supported). Nothing is downloaded here.
from pathlib import Path

def _resolve_preferring_drive(env_var, fallback, dataset_root_env=None):
    """Prefer an explicit env var, then a Drive mount, then the project folder."""
    if dataset_root_env:
        try:
            import colab_helpers as _H
            return _H.resolve_dataset_root()
        except Exception:
            pass
    if env_var:
        candidate = Path(env_var)
        if candidate.exists():
            return candidate
    drive = Path("/content/drive/MyDrive")
    if drive.exists():
        candidate = drive / fallback
        if candidate.exists():
            return candidate
    candidate = PROJECT_ROOT / fallback
    if candidate.exists():
        return candidate
    return None

drive_output = Path("/content/drive/MyDrive/dr_baseline_outputs")

# Persistent output directory: defaults to Drive when Drive is mounted, because a
# Colab runtime is ephemeral. Set DR_OUTPUT_DIR to override.
if os.environ.get("DR_OUTPUT_DIR"):
    OUTPUT_DIR = Path(os.environ["DR_OUTPUT_DIR"])
elif drive_output.parent.exists():
    OUTPUT_DIR = drive_output
else:
    OUTPUT_DIR = PROJECT_ROOT / "dr_baseline_outputs"

SPLIT_DIR = _resolve_preferring_drive(
    os.environ.get("DR_SPLIT_DIR", ""), "provisional_primary_split"
)
DATASET_ROOT = _resolve_preferring_drive(
    os.environ.get("DR_FUNDUS_ROOT", ""),
    "Diabetic Retinopathy and Retinal Diseases Fundus I",
    dataset_root_env=True,
)

print("split dir   :", SPLIT_DIR)
print("dataset root:", DATASET_ROOT)
print("output dir  :", OUTPUT_DIR)
print("output dir is persistent (Drive):", str(OUTPUT_DIR).startswith("/content/drive"))
if SPLIT_DIR is None or DATASET_ROOT is None:
    print("NOTE: set DR_SPLIT_DIR / DR_FUNDUS_ROOT (or mount Drive) before training.")


In [ ]:
# 3. Smoke test - MUST pass before the training cell can run.
# This runs the real pytest suite in colab/smoke_test.py as a subprocess and
# checks its exit status. A non-zero status blocks training.
import subprocess, sys
from pathlib import Path

smoke = PROJECT_ROOT / "colab" / "smoke_test.py"
if not smoke.exists():
    raise FileNotFoundError("colab/smoke_test.py not found in the uploaded project.")

env = os.environ.copy()
if SPLIT_DIR is not None:
    env["DR_SPLIT_DIR"] = str(SPLIT_DIR)
if DATASET_ROOT is not None:
    env["DR_FUNDUS_ROOT"] = str(DATASET_ROOT)

print("Running smoke test:", smoke)
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(smoke), "-q"],
    capture_output=True,
    text=True,
    env=env,
    cwd=str(PROJECT_ROOT),
)

print(result.stdout)
if result.stderr:
    print("stderr:", result.stderr)

SMOKE_OK = result.returncode == 0
print("SMOKE_TEST_RESULT:", "PASS" if SMOKE_OK else "FAIL", "(exit code", result.returncode, ")")
SMOKE_SKIPPED = "skipped" in result.stdout
if SMOKE_SKIPPED:
    print(
        "WARNING: some smoke checks were SKIPPED, so the dataset/split paths were not "
        "fully resolved. Training is still blocked by the path check below, but this "
        "PASS is weaker evidence than a run with zero skips."
    )
if not SMOKE_OK:
    raise SystemExit("Smoke test failed. Do not run the training cell until this passes.")


In [ ]:
# 4. Demonstrate the guard contract WITHOUT training. Safe to run every time.
# This proves the guards are real, rather than only documented in prose.
from colab.train import confirm_guard_contract, device_report
from colab_helpers import BAND_ORDER, pretrained_weights_available, weights_cache_path

print("guard contract           :", json.dumps(confirm_guard_contract(), indent=2))
print("runtime device report    :", json.dumps(device_report(), indent=2))
print("imagenet weights cached  :", pretrained_weights_available())
try:
    print("expected weights cache   :", weights_cache_path())
except Exception as exc:  # torchvision absent from this runtime
    print("expected weights cache   : unavailable (", exc, ")")

assert confirm_guard_contract()["training_requires_confirm"]
assert confirm_guard_contract()["training_requires_smoke_test"]
assert confirm_guard_contract()["final_evaluation_requires_confirm"]

if not pretrained_weights_available():
    print(
        "\nNOTE: ImageNet ResNet-50 weights are not cached yet. Training will refuse to "
        "start unless you pass allow_weight_download=True, which is an explicit "
        "one-time network download decision you make below."
    )

# Download-free architecture check: build ResNet-50 with weights=None and verify the
# frozen-backbone / trainable-head contract plus the output logits shape.
try:
    import torchvision  # noqa: F401
    def _build_check():
        from colab_helpers import (
            assert_head_is_trainable,
            build_frozen_resnet50,
            inspect_model_structure,
        )
        m = build_frozen_resnet50(num_classes=len(BAND_ORDER), pretrained=False)
        assert_head_is_trainable(m)
        info = inspect_model_structure(m)
        with torch.no_grad():
            out = m(torch.zeros(1, 3, 224, 224))
        print('head out_features     :', info['head_out_features'])
        print('trainable parameters  :', len(info['trainable_parameters']))
        print('frozen parameters     :', info['frozen_parameter_count'])
        print('forward output shape  :', tuple(out.shape))
        assert tuple(out.shape) == (1, len(BAND_ORDER)), 'unexpected head output shape'
        return True
    MODEL_CONSTRUCTION_OK = _build_check()
    print('MODEL_CONSTRUCTION_OK :', MODEL_CONSTRUCTION_OK)
except ImportError as exc:
    MODEL_CONSTRUCTION_OK = False
    print('model construction check skipped (torchvision missing):', exc)


## Guarded training

The cell below is disabled by default. It requires **both** the smoke test above to have
passed and an explicit opt-in (`RUN_TRAINING = True`). It never touches the test partition:
the test partition is reserved for the separate evaluation cell further down.


In [ ]:
# 5. Training cell - GUARDED and disabled by default.
#
# To run it:
#   1. Confirm the smoke test cell above printed PASS.
#   2. Set RUN_TRAINING = True below.
#   3. Run this cell.
#
# Anything else stops the cell before a single image is read.

RUN_TRAINING = False

# Explicit, separate consent for the one-time ImageNet weight download.
ALLOW_WEIGHT_DOWNLOAD = False

# Set True only if you deliberately want to replace an existing run directory.
OVERWRITE_OUTPUT = False

if not SMOKE_OK:
    raise SystemExit("Training cell blocked: smoke test did not pass.")
if not RUN_TRAINING:
    print("Training cell skipped. Set RUN_TRAINING = True to run it.")
    raise SystemExit(0)
if SPLIT_DIR is None or DATASET_ROOT is None:
    raise SystemExit(
        "Training cell blocked: DR_SPLIT_DIR and DR_FUNDUS_ROOT must resolve first "
        "(see the path-resolution cell)."
    )

print("Training cell executing (guarded). Provisional baseline - not a clinical result.")

from colab.train import run_training

training_result = run_training(
    split_dir=SPLIT_DIR,
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    patience=3,
    image_size=224,
    seed=12345,
    confirm_training=True,          # explicit opt-in
    smoke_test_passed=SMOKE_OK,     # real smoke-test gate
    allow_weight_download=ALLOW_WEIGHT_DOWNLOAD,
    overwrite=OVERWRITE_OUTPUT,
)

TRAINING_COMPLETED = True
print("training completed")
print("best epoch   :", training_result["best_epoch"])
print("best val loss:", training_result["best_val_loss"])
print("output dir   :", training_result["output_dir"])
print("history      :", len(training_result["final_state"]["train_history"]), "epochs")


In [ ]:
# 6. Final test-partition evaluation - GUARDED and explicitly invoked.
#
# This is the ONLY place the held-out test partition is used. It loads the
# checkpoint chosen on validation loss and reports accuracy, macro precision,
# macro recall, macro F1, per-class metrics/support, and a confusion matrix.

RUN_FINAL_EVALUATION = False

if not SMOKE_OK:
    raise SystemExit("Final evaluation blocked: smoke test did not pass.")
if not RUN_FINAL_EVALUATION:
    print("Final evaluation skipped. Set RUN_FINAL_EVALUATION = True to run it.")
    raise SystemExit(0)
# Blocked unless a validation-selected best checkpoint actually exists on disk, so a
# checkpoint from an earlier session can still be evaluated.
BEST_CKPT = OUTPUT_DIR / "best_checkpoint" / "model.pt"
if not BEST_CKPT.exists():
    raise SystemExit(
        "Final evaluation blocked: no best checkpoint at " + str(BEST_CKPT) + ". "
        "Run the training cell first (this session or a previous one)."
    )
print("best checkpoint:", BEST_CKPT)

from colab.train import run_final_evaluation

eval_result = run_final_evaluation(
    split_dir=SPLIT_DIR,
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    confirm_final_evaluation=True,
)

metrics = eval_result["metrics"]
print("accuracy       :", metrics["accuracy"])
print("macro precision:", metrics["macro_precision"])
print("macro recall   :", metrics["macro_recall"])
print("macro F1       :", metrics["macro_f1"])
print("per class      :", json.dumps(metrics["per_class"], indent=2))
print("confusion matrix:", json.dumps(metrics["confusion_matrix"]))
print("saved to       :", eval_result["output_path"])


## Outputs and interruption behaviour

Everything is written under `OUTPUT_DIR` (Drive by default when Drive is mounted):

- `run_meta.json` - full configuration snapshot, including class weights and the split source
- `run_manifest.json` - run status: `in_progress`, `completed`, or `incomplete`
- `run_summary.json` - per-epoch training/validation history and final validation metrics
- `checkpoints/epoch_XXXX/` - per-epoch `model.pt` and `training_state.json`
- `best_checkpoint/` - the validation-selected best checkpoint
- `final_test_evaluation/final_test_metrics.json` - the separate test-partition report
- `TRAINING_INCOMPLETE.txt` - written only if a run failed or was interrupted

Overwrite behaviour: a run refuses to start if `OUTPUT_DIR` already contains a run.
You must either choose a new directory or pass `overwrite=True` explicitly.

Interruption behaviour: **resume is not implemented.** An interrupted run is marked
`incomplete` and must be restarted from scratch in a fresh output directory. Per-epoch
checkpoints and optimizer state are still saved, so a future resume implementation has the
state it needs.

Provisional status: all reported numbers come from a provisional five-band mapping. They are
not a clinical result and do not imply specialist validation.
